 # STT Evaluation – Kazakh Speech Recognition (OOM‑Safe Subprocess Architecture)

 **Models:** FastConformer, Soyle ONNX, QuartzNet, Whisper‑Turbo‑KSC2

 **Audio folders:**

   - `./data/shala_audio` – Kazakh mixed with Russian (shala speech)

   - `./data/kk_audio`    – Pure Kazakh speech

 **GPU:** RTX 2060 6 GB · **CPU:** i5‑9400



 Each model runs in a **separate subprocess** so that GPU memory is fully released between runs.

 Results are stored as JSON and finally aggregated into CSV.

In [ ]:
# ─── CELL 1: Install dependencies ───────────────────────────────────────────
# Run once. Runtime will restart automatically at the end.
# After restart, skip this cell and run from Cell 2.

import os

# Install everything except torch first
os.system('''pip install -q \
  "nemo_toolkit[asr]" \
  transformers \
  "optimum[onnxruntime]" \
  onnxruntime-gpu==1.19.2 \
  jiwer soundfile huggingface_hub librosa \
  "pandas==2.2.2" \
  "protobuf==5.29.1" \
  "fsspec==2025.3.0" \
  "numpy<2.2" \
  "setuptools>=79.0.0"''')

# Force CUDA torch LAST so NeMo cannot downgrade it
os.system('''pip install -q --force-reinstall --no-deps \
  torch==2.5.1+cu121 \
  torchvision==0.20.1+cu121 \
  torchaudio==2.5.1+cu121 \
  --index-url https://download.pytorch.org/whl/cu121''')

# Install ffmpeg for audio decoding
os.system('apt-get install -q -y ffmpeg libsndfile1')

os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'
print('Done — restarting runtime...')
os.kill(os.getpid(), 9)

 ## 1. Command‑line argument parsing (for worker processes)

In [2]:
# ─── CELL 2: Verify GPU ──────────────────────────────────────────────────────
import os
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'

import warnings
warnings.filterwarnings('ignore')

import torch
print(f'torch:  {torch.__version__}')
print(f'CUDA:   {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:    {torch.cuda.get_device_name(0)}')
    print(f'VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('NO GPU — go to Runtime > Change runtime type > T4 GPU, then re-run Cell 1')

torch:  2.5.1+cu121
CUDA:   True
GPU:    Tesla T4
VRAM:   15.6 GB


 ## 2. Worker process: evaluate a single model

In [3]:
 # ─── CELL 3: Verify audio files exist ──────────────────────────────────────
import os, warnings
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'
warnings.filterwarnings('ignore')

from pathlib import Path

SHALA_DIR = Path('data/shala_audio')
KK_DIR    = Path('data/kk_audio')
SUPPORTED_EXT = {'.wav', '.mp3', '.m4a', '.flac', '.ogg'}

def check_directory(d):
    if not d.exists():
        raise FileNotFoundError(f"Directory not found: {d}")
    files = [f for f in d.glob('*.*') if f.suffix.lower() in SUPPORTED_EXT]
    if not files:
        raise FileNotFoundError(f"No supported audio files in {d}")
    print(f"✓ Found {len(files)} file(s) in {d}: {[f.name for f in files]}")
    return files

print("Checking audio directories...")
shala_files = check_directory(SHALA_DIR)
kk_files    = check_directory(KK_DIR)
print("\nReady to transcribe.")

Checking audio directories...
✓ Found 3 file(s) in data/shala_audio: ['nu-nege.mp3', 'koilarsyndar_medicina.mp3', 'shup_kozge.mp3']
✓ Found 1 file(s) in data/kk_audio: ['tez_bayip_ketem.mp3']

Ready to transcribe.


 ## 3. Main process: run each model in a separate subprocess

In [4]:
# ─── CELL 4: Write worker script (runs one model in a subprocess) ───────────
%%writefile asr_worker.py
import os, sys, json, time, warnings
warnings.filterwarnings('ignore')
import torch
import librosa
from pathlib import Path

# Model name from argument
model_name = sys.argv[1] if len(sys.argv) > 1 else None
if not model_name:
    print("Usage: python asr_worker.py <model>")
    sys.exit(1)

AUDIO_SETS = {
    "shala_audio": Path("data/shala_audio"),
    "kk_audio":    Path("data/kk_audio"),
}
SUPPORTED_EXT = {".wav", ".mp3", ".m4a", ".flac", ".ogg"}

def get_audio_files(directory):
    return sorted([f for f in directory.glob("*.*") if f.suffix.lower() in SUPPORTED_EXT])

def load_audio_and_duration(path, target_sr=16000):
    audio, sr = librosa.load(str(path), sr=target_sr, mono=True)
    duration = len(audio) / sr
    return audio, sr, duration

def evaluate(func):
    results = {}
    for ds_name, ds_path in AUDIO_SETS.items():
        ds_results = []
        for f in get_audio_files(ds_path):
            print(f"  [{ds_name}] {f.name} ...", end=" ", flush=True)
            try:
                text, elapsed, dur = func(f)
                ds_results.append({
                    "file": f.name,
                    "text": text,
                    "elapsed": elapsed,
                    "duration": dur,
                    "rtf": elapsed / dur if dur > 0 else None
                })
                print(f"RTF={elapsed/dur:.3f}")
            except Exception as e:
                print(f"ERROR: {e}")
                ds_results.append({
                    "file": f.name,
                    "text": f"[ERROR: {e}]",
                    "elapsed": 0.0,
                    "duration": 0.0,
                    "rtf": None
                })
        results[ds_name] = ds_results
    out_path = Path("results") / f"{model_name}.json"
    out_path.parent.mkdir(exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"Saved {model_name} results to {out_path}")

# ---------- Model loaders ----------
if model_name == "soyle":
    from transformers import AutoTokenizer, AutoFeatureExtractor
    from optimum.onnxruntime import ORTModelForSpeechSeq2Seq
    import onnxruntime as ort
    provider = "CUDAExecutionProvider" if "CUDAExecutionProvider" in ort.get_available_providers() else "CPUExecutionProvider"
    model = ORTModelForSpeechSeq2Seq.from_pretrained(
        "dhcppc0/soyle_onnx",
        provider=provider,
        encoder_file_name="encoder_model.onnx",
        decoder_file_name="decoder_model.onnx",
        decoder_with_past_file_name="decoder_with_past_model.onnx",
    )
    tokenizer = AutoTokenizer.from_pretrained("dhcppc0/soyle_onnx")
    extractor = AutoFeatureExtractor.from_pretrained("dhcppc0/soyle_onnx")
    device = "cuda" if provider == "CUDAExecutionProvider" else "cpu"
    model.to(device)
    def infer(f):
        audio, sr, dur = load_audio_and_duration(f)
        inputs = extractor(audio, sampling_rate=sr, return_tensors="pt").to(device)
        start = time.time()
        with torch.no_grad():
            generated_ids = model.generate(**inputs, language="<|kk|>", task="transcribe")
        elapsed = time.time() - start
        text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
        return text, elapsed, dur

elif model_name == "fastconformer":
    import nemo.collections.asr as nemo_asr
    model = nemo_asr.models.EncDecHybridRNNTCTCBPEModel.from_pretrained(
        "nvidia/stt_kk_ru_fastconformer_hybrid_large"
    ).cuda().eval()
    def infer(f):
        audio, sr, dur = load_audio_and_duration(f)
        start = time.time()
        result = model.transcribe([audio], batch_size=1)
        elapsed = time.time() - start
        def extract_text(x):
            if isinstance(x, str):
                return x
            if isinstance(x, list) and len(x) > 0:
                return extract_text(x[0])
            if hasattr(x, "text"):
                return x.text
            return str(x)
        text = extract_text(result).strip()
        return text, elapsed, dur

elif model_name == "quartznet":
    import nemo.collections.asr as nemo_asr
    from huggingface_hub import hf_hub_download
    nemo_path = hf_hub_download(
        repo_id="transiteration/stt_kz_quartznet15x5",
        filename="models/stt_kz_quartznet15x5.nemo"
    )
    model = nemo_asr.models.EncDecCTCModel.restore_from(nemo_path).cuda().eval()
    def infer(f):
        audio, sr, dur = load_audio_and_duration(f)
        start = time.time()
        result = model.transcribe([audio], batch_size=1)
        elapsed = time.time() - start
        if isinstance(result, str):
            text = result
        elif isinstance(result, list) and len(result) > 0:
            first = result[0]
            text = first if isinstance(first, str) else (first.text if hasattr(first, "text") else str(first))
        else:
            text = str(result)
        return text.strip(), elapsed, dur

elif model_name == "whisper-turbo":
    from transformers import WhisperProcessor, WhisperForConditionalGeneration
    processor = WhisperProcessor.from_pretrained("abilmansplus/whisper-turbo-ksc2")
    model = WhisperForConditionalGeneration.from_pretrained(
        "abilmansplus/whisper-turbo-ksc2",
        torch_dtype=torch.float16,
        device_map="cuda"
    )
    model.eval()
    def infer(f):
        audio, sr, dur = load_audio_and_duration(f)
        input_features = processor(
            audio, sampling_rate=sr, return_tensors="pt"
        ).input_features.to(device="cuda", dtype=torch.float16)
        start = time.time()
        with torch.no_grad():
            generated_ids = model.generate(input_features)
        elapsed = time.time() - start
        text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
        return text, elapsed, dur

else:
    print(f"Unknown model: {model_name}")
    sys.exit(1)

evaluate(infer)

Writing asr_worker.py


 ## 4. Aggregation: combine JSON results into CSV

In [5]:
# ─── CELL 5: Run models sequentially (each in its own subprocess) ───────────
import subprocess
import sys
import time
from pathlib import Path

models = ["soyle", "fastconformer", "quartznet", "whisper-turbo"]
print("Running evaluation for all models (each in its own subprocess)...")

for model in models:
    print(f"\n--- Launching {model} ---")
    # Use sys.executable to run the worker script
    subprocess.run([sys.executable, "asr_worker.py", model], check=True)
    time.sleep(2)  # extra safety to release GPU memory

print("\nAll models finished.")

Running evaluation for all models (each in its own subprocess)...

--- Launching soyle ---

--- Launching fastconformer ---

--- Launching quartznet ---

--- Launching whisper-turbo ---

All models finished.


In [6]:
# ─── CELL 6: Aggregate JSON results into CSV ────────────────────────────────
import json
import pandas as pd
from pathlib import Path
from google.colab import files

results_dir = Path("results")
if not results_dir.exists():
    print("No results directory found. Run Cell 5 first.")
else:
    all_rows = []
    for json_file in results_dir.glob("*.json"):
        model = json_file.stem
        with open(json_file, "r", encoding="utf-8") as f:
            data = json.load(f)
        for ds_name, items in data.items():
            for item in items:
                all_rows.append({
                    "Dataset": ds_name,
                    "Audio": item["file"],
                    "Model": model,
                    "Transcription": item["text"],
                    "Processing Time (s)": item["elapsed"],
                    "Duration (s)": item["duration"],
                    "RTF": item["rtf"],
                })
    if not all_rows:
        print("No data found.")
    else:
        df = pd.DataFrame(all_rows)
        # Pivot to wide format: one row per audio file, columns per model
        pivot = df.pivot_table(
            index=["Dataset", "Audio", "Duration (s)"],
            columns="Model",
            values=["Transcription", "RTF"],
            aggfunc="first"
        )
        pivot.columns = [f"{col[0]}_{col[1]}" for col in pivot.columns]
        pivot = pivot.reset_index()
        csv_path = "transcriptions_combined.csv"
        pivot.to_csv(csv_path, index=False)
        print(f"Saved combined CSV: {csv_path}")
        print("\n--- Preview ---")
        print(pivot.head())
        files.download(csv_path)

Saved combined CSV: transcriptions_combined.csv

--- Preview ---
       Dataset                      Audio  Duration (s)  RTF_fastconformer  \
0     kk_audio        tez_bayip_ketem.mp3     53.545250           0.006261   
1  shala_audio  koilarsyndar_medicina.mp3     60.139688           0.018699   
2  shala_audio                nu-nege.mp3     63.808437           0.006784   
3  shala_audio             shup_kozge.mp3     27.422812           0.006314   

   RTF_quartznet  RTF_soyle  RTF_whisper-turbo  \
0       0.001538   0.094208           0.029728   
1       0.008118   0.095083           0.038742   
2       0.001727   0.057851           0.020927   
3       0.002448   0.096461           0.042471   

                         Transcription_fastconformer  \
0  (['көп адамдар осы жерден шатасады ез байып ке...   
1  (['ііі сізге қоятын сұрақтарым бар м кой кой ы...   
2  (['медицина саласында бюджеттен жылына қанша а...   
3  (['сіз қалайсыз қалайсыз деп қояды ғой көріп т...   

            

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>